# 📰  Pydantic 파이프라인 + 뉴스 모니터링

```
Phase 0~4. Core 학습 — Pydantic + RunnableParallel 핵심 부품
Project.   기사 → 분석 + 요약 + 태깅 + 대응 라우팅
```

## Phase 0. 환경 설정

In [29]:
%pip install -q langchain langchain-openai python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

✅ 로컬 Jupyter
✅ API 키 OK


## Phase 1. PydanticOutputParser — 자연어를 객체로

5단계 패턴: 스키마 → 파서 → 프롬프트 → LLM → 체인

In [31]:
class Summary(BaseModel):
    title: str = Field(description="짧은 제목")
    sentiment: Literal["positive", "negative", "neutral"]
    key_points: list[str] = Field(description="핵심 2-3개")

parser = PydanticOutputParser(pydantic_object=Summary)

prompt = ChatPromptTemplate.from_messages([
    ("system", "텍스트를 분석합니다. 한국어로.\n\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain_summary = prompt | llm | parser

r = chain_summary.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r)

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시일이 6월에서 7월로 연기됨']


## Phase 2. RunnableParallel — 여러 체인 결합

In [32]:
# 자연어 답장 체인 추가 (StrOutputParser)
chain_notice = (
    ChatPromptTemplate.from_messages([
        ("system", "팀원에게 알릴 한 줄 안내문을 작성하세요."),
        ("human", "{text}"),
    ]) | llm | StrOutputParser()
)

combined = RunnableParallel(summary=chain_summary, notice=chain_notice)

r = combined.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r["summary"])
print(r["notice"])

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시일이 6월에서 7월로 연기됨']
신제품 출시가 6월에서 7월로 연기되었음을 알려드립니다.


## Phase 3. RunnableLambda — 함수도 체인에 (결과는 Pydantic으로)

In [33]:
class Decision(BaseModel):
    urgent: bool
    note: str

def route(x) -> Decision:
    s = x["summary"]
    return Decision(urgent=(s.sentiment == "negative"), note=s.title)

chain_routed = combined | RunnableLambda(route)

d = chain_routed.invoke({"text": "신제품 출시 연기"})
print(d)

urgent=True note='신제품 출시 연기'


## Phase 4. RunnablePassthrough — 원본 보존

In [34]:
full = RunnableParallel(
    summary=chain_summary,
    notice=chain_notice,
    original=RunnablePassthrough(),
)
r = full.invoke({"text": "신제품 출시 연기"})
print(r["original"])  # 입력 그대로 통과

{'text': '신제품 출시 연기'}


---
# 📰 Project. 뉴스 모니터링 + PR 대응 시스템

```
기사 → 분석 + 임원 요약 + 태깅 (동시)
     → 대응 라우팅 (즉시 알림 / 일일 브리핑 / 보관)
```

**모니터링 대상**: NovaTech (가상 AI 스타트업, B2B 챗봇 솔루션)

In [35]:
SAMPLE = """[테크크런치] NovaTech, B2B 챗봇 시장 점유율 1위 등극.
LLM 챗봇을 금융·제조 대기업 50여 곳에 공급하며 전년 대비 200% 성장.
다만 일부 고객사는 응답 속도 개선이 필요하다고 지적."""

# 분석 체인 — 자사 관점에서
class NewsAnalysis(BaseModel):
    headline: str = Field(description="15자 이내 헤드라인")
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    impact_score: int = Field(ge=0, le=10, description="자사 영향도 0-10")
    contains_risk: bool = Field(description="평판/사업 리스크 언급 여부")

p1 = PydanticOutputParser(pydantic_object=NewsAnalysis)
chain_analysis = (
    ChatPromptTemplate.from_messages([
        ("system", "자사(NovaTech, B2B AI 챗봇 스타트업) 관점에서 기사를 분석합니다.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p1.get_format_instructions())
    | llm | p1
)
print(chain_analysis.invoke({"article": SAMPLE}))

headline='NovaTech, B2B 챗봇 시장 1위' sentiment='positive' impact_score=8 contains_risk=True


In [36]:
# 임원 보고용 3문장 요약 (자연어)
chain_brief = (
    ChatPromptTemplate.from_messages([
        ("system", "임원 보고용 *정확히 3문장* 요약: 사실 / 자사 관련성 / 시사점."),
        ("human", "{article}"),
    ]) | llm | StrOutputParser()
)
print(chain_brief.invoke({"article": SAMPLE}))

NovaTech는 B2B 챗봇 시장에서 1위로 등극하며, 금융 및 제조 대기업 50여 곳에 LLM 챗봇을 공급하여 전년 대비 200% 성장했다. 이는 자사의 기술력이 시장에서 높은 평가를 받고 있음을 보여준다. 그러나 일부 고객의 응답 속도 개선 요구는 향후 서비스 품질 향상에 대한 필요성을 시사한다.


In [37]:
# 태깅 체인 — 다중 선택 (list[Literal])
class NewsTags(BaseModel):
    topics: list[Literal["제품", "재무", "인사", "기술", "규제", "경쟁사", "기타"]]
    teams: list[Literal["PR", "마케팅", "프로덕트", "법무", "재무"]] = Field(
        description="이 기사를 봐야 할 사내 팀"
    )

p2 = PydanticOutputParser(pydantic_object=NewsTags)
chain_tags = (
    ChatPromptTemplate.from_messages([
        ("system", "기사에 적절한 주제와 관련 팀 태그.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p2.get_format_instructions())
    | llm | p2
)

# 3개 체인 동시 실행
monitor_v1 = RunnableParallel(
    analysis=chain_analysis,
    brief=chain_brief,
    tags=chain_tags
)

r = monitor_v1.invoke({"article": SAMPLE})
print(f"📌 {r['analysis'].headline} | 영향도 {r['analysis'].impact_score}")
print(f"🏷  {r['tags'].topics} / 팀: {r['tags'].teams}")

📌 NovaTech, B2B 챗봇 시장 1위 | 영향도 8
🏷  ['기술', '경쟁사'] / 팀: ['마케팅', '프로덕트']


In [38]:
# 대응 라우팅 — RunnableLambda + Pydantic
class PRAction(BaseModel):
    priority: Literal["alert_now", "daily_brief", "archive"]
    channel: Literal["slack_urgent", "slack_daily", "notion"]
    message: str

def route_pr(x) -> PRAction:
    a = x["analysis"]
    if a.sentiment == "negative" and a.impact_score >= 7:
        return PRAction(priority="alert_now", channel="slack_urgent",
                       message=f"🚨 즉시 대응: {a.headline}")
    if a.impact_score >= 4:
        return PRAction(priority="daily_brief", channel="slack_daily",
                       message=f"📋 브리핑: {a.headline}")
    return PRAction(priority="archive", channel="notion",
                   message=f"📁 보관: {a.headline}")

monitor_v2 = monitor_v1 | RunnableLambda(route_pr)
print(monitor_v2.invoke({"article": SAMPLE}))

priority='daily_brief' channel='slack_daily' message='📋 브리핑: NovaTech, B2B 챗봇 시장 1위'


In [39]:
# 여러 기사 batch — 매일 들어오는 수십 건
ARTICLES = [
    {"article": SAMPLE},
    {"article": "NovaTech 일부 고객사 데이터 유출 의혹. 회사는 부인. 보안 업계 우려 제기."},
    {"article": "정부, AI 산업 5조원 지원금 편성. 스타트업·연구기관 대상."},
]
for a in monitor_v2.batch(ARTICLES):
    icon = {"alert_now": "🚨", "daily_brief": "📋", "archive": "📁"}[a.priority]
    print(f"{icon} {a.message}")

📋 📋 브리핑: NovaTech, B2B 챗봇 시장 1위
🚨 🚨 즉시 대응: 고객사 데이터 유출 의혹
📋 📋 브리핑: 정부, AI 산업 지원


---
## 🎓 마무리

같은 3개 부품으로 PR 모니터링 시스템 완성. **도메인이 달라도 패턴은 동일**.

**도전과제**: SNS 카피 체인 / 영문 번역 체인 / 경쟁사 비교 체인 추가


### 경쟁사 비교 체인 추가

In [42]:
# 경쟁사 비교 체인 — 기사 속 경쟁 구도를 자사(NovaTech) 기준으로 비교
# 1) 스키마: LLM이 채워야 할 비교 항목을 필드로 정의
class CompareRival(BaseModel):
    competitors: list[str] = Field(description="기사에 언급되거나 연관된 경쟁사 (없으면 빈 리스트)")
    our_strengths: list[str] = Field(description="경쟁사 대비 자사 강점")
    our_weaknesses: list[str] = Field(description="경쟁사 대비 자사 약점")
    position: Literal["우위", "열세", "비슷", "판단불가"]  # Literal → 정해진 4개 값 중 하나만 허용
    action_point: str = Field(description="경쟁 대응 한 줄 제안")

# 2) 파서: LLM 응답(JSON 문자열)을 CompareRival 객체로 변환
compare_parser = PydanticOutputParser(pydantic_object=CompareRival)

# 3) 프롬프트 → 4) LLM → 5) 파서 순서로 연결
chain_compare = (
    ChatPromptTemplate.from_messages([
        ("system", "자사(NovaTech, B2B AI 챗봇 스타트업)의 경쟁사 관점에서 비교분석합니다.{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=compare_parser.get_format_instructions())  # 출력 형식 안내문을 미리 채워 둠
    | llm | compare_parser
)

# 단독 테스트 — 결과는 CompareRival 객체
print(chain_compare.invoke({"article": SAMPLE}))

competitors=['ChatGPT', 'IBM Watson', 'Zendesk', 'LivePerson'] our_strengths=['B2B 시장에서의 높은 점유율', '50여 개 대기업에 공급 경험', '200% 성장률'] our_weaknesses=['응답 속도 개선 필요'] position='우위' action_point='응답 속도 개선을 위한 기술 투자 및 최적화 진행'


In [44]:
# SNS 카피 체인 — 기사를 SNS 게시용 한 문장 + 해시태그로 변환
# 1) 스키마
class SNSCopy(BaseModel):
    text: str = Field(description="SNS용 1문장 안내문")
    hashtags: list[str] = Field(description="관련 해시태그 2-3개")

# 2) 파서
parser_sns = PydanticOutputParser(pydantic_object=SNSCopy)

# 3) 프롬프트 — format_instructions를 partial로 미리 채움
prompt_sns = ChatPromptTemplate.from_messages([
    ("system", "SNS용 안내문을 작성합니다. 한국어로.{format_instructions}"),
    ("human", "{article}"),
]).partial(format_instructions=parser_sns.get_format_instructions())

# 4) LLM — 카피는 표현이 다양해야 하므로 분석용(temperature=0)과 별도로 0.5 사용
llm_sns = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

# 5) 체인 구성 (프롬프트 → LLM → 파서 순차 연결)
chain_sns = prompt_sns | llm_sns | parser_sns

# batch: 여러 입력을 리스트로 넘기면 동시에 처리해 결과도 리스트로 반환
#        (기사 3건을 invoke 3번 돌리는 것보다 빠름)
result_sns = chain_sns.batch([
    {"article": "신제품 출시 연기"}
    ,{"article": "NovaTech, B2B 챗봇 시장 점유율 1위 등극"}
    ,{"article": "1팀 데이터 유출 의혹, 팀장은 부인"}
])
print(result_sns)

[SNSCopy(text='신제품 출시가 연기되었습니다. 더 나은 품질을 위해 노력하겠습니다.', hashtags=['#신제품', '#출시연기', '#품질향상']), SNSCopy(text='NovaTech가 B2B 챗봇 시장에서 점유율 1위에 올랐습니다!', hashtags=['#NovaTech', '#B2B챗봇', '#시장점유율']), SNSCopy(text='1팀 데이터 유출 의혹, 팀장은 부인', hashtags=['#데이터유출', '#팀장', '#의혹'])]


In [45]:
# 번역 체인 — 기사 원문을 영문으로 번역
# (참고) 구조화 출력용 스키마 — 현재 체인은 StrOutputParser를 써서 이 클래스는 사용하지 않음
class Translation(BaseModel):
    original : str = Field(description="원문")
    translated : str = Field(description="영문 번역문")

# 번역문은 문자열 하나면 충분하므로 StrOutputParser로 텍스트만 받음
translation_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Translate the following text into English, keeping the meaning intact."),
        ("human", "{article}"),
    ]) | llm | StrOutputParser()
)

# 입력은 프롬프트 변수명과 같은 키를 가진 dict로 전달
print(translation_chain.invoke({"article": SAMPLE}))

[TechCrunch] NovaTech has risen to the top of the B2B chatbot market share. The company has supplied LLM chatbots to over 50 large enterprises in the finance and manufacturing sectors, achieving a 200% growth compared to the previous year. However, some clients have pointed out the need for improvements in response speed.


### 합본

In [46]:
# 6개 체인 동시 실행 — 같은 {"article"} 입력을 각 체인에 동시에 넘기고
# 결과를 키 이름별 dict로 모음: {"analysis": ..., "brief": ..., ..., "compare": ...}
monitor_new = RunnableParallel(
    analysis=chain_analysis,        # 분석 (NewsAnalysis)
    brief=chain_brief,              # 임원 보고 3문장 요약 (str)
    tags=chain_tags,                # 주제/팀 태깅 (NewsTags)
    translation=translation_chain,  # 영문 번역 (str)
    sns = chain_sns,                # SNS 카피 (SNSCopy)
    compare=chain_compare           # 경쟁사 비교 (CompareRival)
)

# 6개 결과는 유지 + 대응 라우팅 결과를 "action" 키로 추가
# - monitor_new | RunnableLambda(route_pr) 로 바로 연결하면 PRAction만 남고 6개 결과가 사라짐
# - RunnablePassthrough.assign()은 들어온 dict를 그대로 통과시키면서 새 키만 덧붙임
monitor_full = monitor_new | RunnablePassthrough.assign(action=RunnableLambda(route_pr))

r = monitor_full.invoke({"article": SAMPLE})

# 체인별 결과 확인 — Pydantic 객체는 .필드명, 문자열은 그대로 출력
print(f"📌 {r['analysis'].headline} | 영향도 {r['analysis'].impact_score}")
print(f"🏷  {r['tags'].topics} / 팀: {r['tags'].teams}")
print(f"🌐 {r['translation']}")
print(f"📱 {r['sns'].text} {' '.join(r['sns'].hashtags)}")
print(f"🤝 {r['compare'].position} | 경쟁사: {r['compare'].competitors} | {r['compare'].action_point}")
print(f"➡️  {r['action'].channel} | {r['action'].message}")

📌 NovaTech, B2B 챗봇 시장 1위 | 영향도 8
🏷  ['기술', '제품'] / 팀: ['마케팅', '프로덕트']
🌐 [TechCrunch] NovaTech has risen to the top of the B2B chatbot market share. The company has supplied LLM chatbots to over 50 large enterprises in the finance and manufacturing sectors, achieving a 200% growth compared to the previous year. However, some clients have pointed out the need for improvements in response speed.
📱 [테크크런치] NovaTech, B2B 챗봇 시장 점유율 1위 등극. LLM 챗봇을 금융·제조 대기업 50여 곳에 공급하며 전년 대비 200% 성장. 다만 일부 고객사는 응답 속도 개선이 필요하다고 지적. #NovaTech #챗봇 #B2B
🤝 우위 | 경쟁사: ['OpenAI', 'Google Dialogflow', 'IBM Watson', 'Microsoft Bot Framework'] | 응답 속도 개선을 위한 기술 투자 및 고객 피드백 반영
➡️  slack_daily | 📋 브리핑: NovaTech, B2B 챗봇 시장 1위
